# Studio 3.4 — Sensitivity Analysis: What Is One More Hour Worth?

**OPIM 5641 · Business Decision Modeling · Dr. Dave Wanik, UConn**

Fitting the model is the easy part. **Interrogating** it is the job.

Back to Flair's Furniture. At the optimum, all 2,400 carpentry hours were used up - carpentry is a
**binding constraint**, a bottleneck. So a savvy analyst asks: *what if I could get a few more
carpentry hours? How much is one more hour actually worth?* That number is the **shadow price**.

Here is a very transparent way to find it: **take your model and stick it in a for loop.** The thing
you change every time is the right-hand side of the binding constraint.

**How this notebook works.** All of the code is already here. You do not have to write a model
from scratch. Your job is to **run it, read it, and change it**: each experiment has a clearly marked
cell where you plug in a scenario, re-run, and see what happens.

## Import modules (download Pyomo)

Run this first, every time. It installs Pyomo and the `cbc` solver in Colab.

In [ ]:
# import modules

# this makes beautiful plots using pylab (matplotlib)
%matplotlib inline
from pylab import * # * means import ALL NAME SPACES
import pandas as pd

# useful modules for downloading pyomo onto Colab
import shutil
import sys
import os.path

# install pyomo if it doesn't exist
if not shutil.which("pyomo"):
    !pip install -q pyomo
    assert(shutil.which("pyomo"))

# install the 'cbc' solver if it doesn't exist
if not (shutil.which("cbc") or os.path.isfile("cbc")):
    if "google.colab" in sys.modules:
        !apt-get install -y -qq coinor-cbc
    else:
        try:
            !conda install -c conda-forge coincbc
        except:
            pass

# make sure 'cbc' is the solver that we will invoke later on
assert(shutil.which("cbc") or os.path.isfile("cbc"))

# import ALL NAMESPACES (variables) from pyomo
from pyomo.environ import *

# Part A · Carpentry hours, with the hard-coded model

Buffer out a little: go backwards from 2,400, and explore well beyond it.

In [ ]:
myHours = np.arange(1200, 3300, 100)
print(myHours)

Now the loop. It is the hard-coded Flair's model from notebook 3.1, with **one change**: the `2400`
became `a`.

In [ ]:
# store the results
myResults = pd.DataFrame() # empty place to store results

for a in myHours:
  # declare the model
  model = ConcreteModel()

  # declare decision variables
  model.tables = Var(domain=NonNegativeReals)
  model.chairs = Var(domain=NonNegativeReals)

  # declare objective
  model.profit = Objective(
                        expr = 7*model.tables + 5*model.chairs,
                        sense = maximize)

  # declare constraints - carpentry hours are now 'a', not 2400
  model.constraint1 = Constraint(expr = 3*model.tables + 4*model.chairs <= a) # carpentry hours
  model.constraint2 = Constraint(expr = 2*model.tables + 1*model.chairs <= 1000) # painting hours
  model.constraint3 = Constraint(expr = model.chairs <= 450) # chair limit
  model.constraint4 = Constraint(expr = model.tables >= 100) # table minimum

  # solve it (no .write() here, or you get 21 printouts!)
  SolverFactory('cbc', executable='/usr/bin/cbc').solve(model)

  # results
  myX = pd.DataFrame([a, model.profit(), model.tables(), model.chairs()])
  myX = myX.T # transpose

  # store the results
  myResults = pd.concat([myResults, myX])

# Change the Column names
myResults = myResults.rename({0:"Carpentry Hours", 1:"Profit", 2:"Tables", 3:"Chairs"}, axis='columns')
myResults.reset_index(drop=True, inplace=True)
myResults

## Product mix

Read the table before you plot it. As the carpentry hours go up, profit goes up - and the **mix**
changes. Let's plot the product mix as a function of carpentry hours.

In [ ]:
# make a nice plot
tmp = myResults.drop(['Profit'], axis=1)
tmp.plot(x='Carpentry Hours')
title('Product mix')
show()

With very few carpentry hours we only make tables (they earn more per carpentry hour). As hours
are added we trade tables for chairs, until we hit the 450-chair limit and the mix stops changing.

## Profit and diminishing returns

Total profit is misleading - of course you make more money as you get more hours. The question is how
much **each additional hour** is worth. Subtract one row from the next and divide by the step.

In [ ]:
# what is ONE more carpentry hour worth?
myResults['ProfitDiffOneHour'] = myResults['Profit'].diff() / myResults['Carpentry Hours'].diff()

# show the table
myResults

In [ ]:
# make a nice plot
myResults.plot(x='Carpentry Hours', y='ProfitDiffOneHour')
title('What is one more carpentry hour worth?')
show()

Read the plot as a staircase:

- **Up to 1,500 hours:** carpentry is so scarce we only make tables, and each extra hour is worth
  about **\$2.33** (one more third of a table, at \$7 each).
- **From 1,500 to about 2,600 hours:** each extra hour is worth exactly **\$0.60**. That's the
  **shadow price** of carpentry at our optimum.
- **Beyond that:** we hit the 450-chair limit, and more carpentry is worth **\$0**.

Your carpentry hours start out like **diamonds** and turn into **sand**: once they're not scarce
anymore, they have no value, because the bottleneck has moved somewhere else in the shop.

**The business question:** suppose an extra carpentry hour costs Flair **\$0.50** in overtime. How many
carpentry hours would you advocate for?

**Tie it back to Simplex:** if you took Flair's all the way to the final tableau, look at the bottom
row under the carpentry slack: **3/5 = 0.60**. The bottom row of the final tableau *is* the shadow
prices.

# Part B · Any resource you like, with the soft-coded model

Here is the payoff from soft-coding. Because the hours live in a dictionary, we can point the loop at
**any** department by changing one word.

Change `resource` and `myRange` in the first cell, then run both cells.

In [ ]:
# ---------- THE SCENARIO (change things here) ----------
resource = "painting"                  # which department to investigate: "carpentry" or "painting"
myRange = np.arange(400, 2100, 100)    # the hours to try (start, stop, step)

In [ ]:
# ---------- THE LOOP (you never change this cell) ----------
myResults2 = pd.DataFrame()

for a in myRange:
    # the data
    products = ["tables", "chairs"]
    profit = {"tables" : 7, "chairs" : 5}
    hours = {
        "carpentry" : {"tables" : 3, "chairs" : 4},
        "painting"  : {"tables" : 2, "chairs" : 1}
    }
    total_time = {"carpentry" : 2400, "painting" : 1000}
    max_units = {"chairs" : 450}
    min_units = {"tables" : 100}

    # the ONE thing we change each time through the loop
    total_time[resource] = a

    # the model (same as notebook 3.1)
    model = ConcreteModel()
    model.x = Var(products, domain=NonNegativeReals)
    obj_expr = 0
    for product in products:
        obj_expr += profit[product] * model.x[product]
    model.profit = Objective(expr = obj_expr, sense = maximize)
    model.constraints = ConstraintList()
    for dept in total_time:
        dept_expr = 0
        for product in products:
            dept_expr += hours[dept][product] * model.x[product]
        model.constraints.add(dept_expr <= total_time[dept])
    for product in max_units:
        model.constraints.add(model.x[product] <= max_units[product])
    for product in min_units:
        model.constraints.add(model.x[product] >= min_units[product])

    SolverFactory('cbc', executable='/usr/bin/cbc').solve(model)

    # store the results
    myX = pd.DataFrame([a, model.profit(), model.x["tables"](), model.x["chairs"]()])
    myX = myX.T
    myResults2 = pd.concat([myResults2, myX])

myResults2 = myResults2.rename({0:"Hours", 1:"Profit", 2:"Tables", 3:"Chairs"}, axis='columns')
myResults2.reset_index(drop=True, inplace=True)
myResults2['ProfitDiffOneHour'] = myResults2['Profit'].diff() / myResults2['Hours'].diff()

myResults2.plot(x='Hours', y='ProfitDiffOneHour')
title('What is one more ' + resource + ' hour worth?')
show()
myResults2

**Your turn:**

1. What is one more **painting** hour worth at the optimum (1,000 hours)? Check it against your
   final Simplex tableau.
2. At what point do painting hours turn from diamonds into sand?
3. If you could buy only one extra hour, would you buy carpentry or painting? Why?

# Part C · The price of salt at McDonald's

The same trick works on any model. In notebook 3.3, the sodium limit was a binding rule. **What does
each 500 mg of sodium cost the team?**

Change `nutrient`, `rule` and `myRange` in the first cell if you want to investigate a different rule.

In [ ]:
# read the McDonald's data (same as notebook 3.3)
data_folder = 'https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio3/menus/'
menu_file = 'mcdonalds_menu.csv'
menu_df = pd.read_csv(data_folder + menu_file)
rules_df = pd.read_csv(data_folder + 'team_rules.csv')

items = list(menu_df['item'])
nutrient_columns = list(menu_df.columns[3:])

price = {}
nutrition = {}
for i in range(len(menu_df)):
    item = menu_df.loc[i, 'item']
    price[item] = float(menu_df.loc[i, 'price'])
    nutrition[item] = {}
    for nutrient in nutrient_columns:
        nutrition[item][nutrient] = float(menu_df.loc[i, nutrient])

player_min = {}
player_max = {}
for i in range(len(rules_df)):
    nutrient = rules_df.loc[i, 'nutrient']
    if rules_df.loc[i, 'rule'] == 'min':
        player_min[nutrient] = int(rules_df.loc[i, 'per_player'])
    else:
        player_max[nutrient] = int(rules_df.loc[i, 'per_player'])

print("minimums per player:", player_min)
print("maximums per player:", player_max)

In [ ]:
# ---------- THE SCENARIO (change things here) ----------
nutrient_to_change = "sodium"           # which rule to investigate
rule = "max"                            # is it a "max" rule or a "min" rule?
myRange = np.arange(4500, 8500, 500)    # the limits to try (start, stop, step)
team_size = 53
max_of_one_item = 4

In [ ]:
# ---------- THE LOOP (you never change this cell) ----------
myResults3 = pd.DataFrame()

for a in myRange:
    low = dict(player_min)
    high = dict(player_max)

    # the ONE thing we change each time through the loop
    if rule == "max":
        high[nutrient_to_change] = a
    else:
        low[nutrient_to_change] = a

    # the model (same as notebook 3.3)
    model = ConcreteModel()
    model.x = Var(items, domain=NonNegativeReals)
    obj_expr = 0
    for item in items:
        obj_expr += price[item] * model.x[item]
    model.cost = Objective(expr = obj_expr, sense = minimize)
    model.constraints = ConstraintList()
    for nutrient in low:
        nutrient_expr = 0
        for item in items:
            nutrient_expr += nutrition[item][nutrient] * model.x[item]
        model.constraints.add(nutrient_expr >= low[nutrient] * team_size)
    for nutrient in high:
        nutrient_expr = 0
        for item in items:
            nutrient_expr += nutrition[item][nutrient] * model.x[item]
        model.constraints.add(nutrient_expr <= high[nutrient] * team_size)
    for item in items:
        model.constraints.add(model.x[item] <= max_of_one_item * team_size)

    results = SolverFactory('cbc', executable='/usr/bin/cbc').solve(model)

    # store the results (if there is no feasible order, store a blank)
    if results.solver.termination_condition == TerminationCondition.optimal:
        cost_per_player = model.cost() / team_size
    else:
        cost_per_player = np.nan
    myX = pd.DataFrame([a, cost_per_player])
    myX = myX.T
    myResults3 = pd.concat([myResults3, myX])

myResults3 = myResults3.rename({0:"Limit", 1:"Cost per player"}, axis='columns')
myResults3.reset_index(drop=True, inplace=True)

myResults3.plot(x='Limit', y='Cost per player', marker='o')
title('Cost per player as the ' + nutrient_to_change + ' ' + rule + ' changes')
show()
myResults3

**Your turn:**

1. How much does the bill per player go up when the sodium limit drops from 6,500 mg to 4,500 mg?
2. Above what sodium limit does more salt stop saving money? (Diamonds to sand again.)
3. Pick a different rule and investigate it. For example: `nutrient_to_change = "protein"`,
   `rule = "min"`, `myRange = np.arange(100, 260, 20)`. What does each extra 20 g of protein cost?

## Bottom line

- A **binding constraint** is a bottleneck. Its **shadow price** is what one more unit is worth.
- You do not need anything fancy to find it: **put the model in a for loop**, change the right-hand
  side, and look at the difference from row to row.
- Shadow prices do not last forever. Scarce things are valuable; once a resource is no longer scarce,
  it is worth nothing and the bottleneck is somewhere else. **Diamonds to sand.**
- Soft-coding made Part B and Part C a one-word change.

**Before you leave:** File → Save a copy in GitHub.